# Notebook 2 / R2 - Focus-RCNet KD Teacher-Assistant Selection

Notebook ini menjalankan R2 untuk memilih kandidat Focus-RCNet teacher assistant.
Default saat ini adalah final `ce_baseline` seed 42 sebagai Focus-RCNet control.
Setelah CE baseline final selesai untuk 5 seed, jalankan final `direct_kd`
dengan `T=4` dan `alpha=0.1` sebagai best KD candidate. Split tidak dibuat ulang; semua row train/validation/test dibaca
dari artefak R0.

Rules:
- Gunakan split manifest dari R0.
- Gunakan checkpoint R1 EfficientNet-B4 final sesuai seed sebagai teacher.
- Pilot hanya memakai train dan validation; independent test tidak dievaluasi.
- Pilih `direct_kd` vs `twostage_kd` berdasarkan validation metric.
- Output pilot disimpan di `final_research_kd/runs/pilots/R2/<setup_id>/`.

In [ ]:
# ============================================================
# 1. Imports and Configuration
# ============================================================

from datetime import datetime, timezone
from pathlib import Path
import copy
import json
import os
import random
import time
import warnings

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
warnings.filterwarnings("ignore", category=UserWarning)


class Config:
    EXPERIMENT_ID = "R2"
    EXPERIMENT_NAME = "R2 Focus-RCNet KD Teacher-Assistant Selection"
    PLATFORM = "Kaggle Notebooks"
    FRAMEWORK = "PyTorch"
    USE_AMP = True
    MULTI_GPU = False

    SEED = 3407
    RUN_PHASE = "final"  # "pilot" for R2 selection, "final" after variant is frozen
    R2_VARIANT = "direct_kd"  # "ce_baseline", "direct_kd", or "twostage_kd"

    DATASET_NAME = "TrashNet"
    DATASET_DIR = Path(os.environ.get(
        "TRASHNET_DATASET_DIR",
        "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
    ))

    DEFAULT_OUTPUT_ROOT = (
        Path("/kaggle/working/final_research_kd/runs")
        if Path("/kaggle/working").exists()
        else Path.cwd() / "final_research_kd" / "runs"
    )
    OUTPUT_ROOT = Path(os.environ.get("R2_OUTPUT_ROOT", str(DEFAULT_OUTPUT_ROOT)))

    IMG_SIZE = 380
    BATCH_SIZE = 16
    NUM_WORKERS = 2

    PILOT_CE_EPOCHS = 60
    PILOT_KD_EPOCHS = 60
    FINAL_CE_EPOCHS = 100
    FINAL_KD_EPOCHS = 100
    CE_EPOCHS = PILOT_CE_EPOCHS if RUN_PHASE == "pilot" else FINAL_CE_EPOCHS
    KD_EPOCHS = PILOT_KD_EPOCHS if RUN_PHASE == "pilot" else FINAL_KD_EPOCHS

    DIRECT_LR = 0.05
    STAGE1_LR = 0.05
    STAGE2_LR = 0.001
    MOMENTUM = 0.9
    WEIGHT_DECAY = 1e-4
    SCHEDULER = "CosineAnnealingLR"

    KD_TEMPERATURE = 4.0
    KD_ALPHA = 0.1
    TEACHER_MODEL_NAME = "EfficientNet-B4"
    TEACHER_TIMM_NAME = "efficientnet_b4"
    STUDENT_MODEL_NAME = "Focus-RCNet"

    CHECKPOINT_METRIC = "best_val_accuracy"
    EARLY_STOPPING = RUN_PHASE == "pilot"
    EARLY_STOPPING_MONITOR = "val_loss"
    PATIENCE = 15
    EVALUATE_TEST = RUN_PHASE == "final"

    R0_DIR_CANDIDATES = [
        Path(os.environ["R0_DATA_PROTOCOL_DIR"]) if os.environ.get("R0_DATA_PROTOCOL_DIR") else None,
        Path("/kaggle/input/notebook0-r0-final-data-protocol-setup/final_research/r0_data_protocol"),
        Path("/kaggle/input/notebook0-r0-final-data-protocol-setup/final_research"),
        Path("/kaggle/input/notebook0-r0-final-data-protocol-setup"),
        Path("/kaggle/input/notebook0_r0_final_data_protocol_setup/final_research/r0_data_protocol"),
        Path("/kaggle/input/notebook0_r0_final_data_protocol_setup/final_research"),
        Path("/kaggle/input/notebook0_r0_final_data_protocol_setup"),
        Path("/kaggle/input/notebooks/hamzapratama/notebook0-r0-final-data-protocol-setup/final_research/r0_data_protocol"),
        Path("/kaggle/input/notebooks/hamzapratama/notebook0-r0-final-data-protocol-setup"),
        Path("/kaggle/input/r0-kaggle-final/r0_data_protocol"),
        Path("/kaggle/working/final_research/r0_data_protocol"),
        Path.cwd() / "final_research_kd" / "r0_data_protocol",
        Path.cwd() / "final_research_kd" / "runs" / "r0" / "final_research" / "r0_data_protocol",
    ]
    R1_TEACHER_ROOT_CANDIDATES = [
        Path(os.environ["R1_TEACHER_DIR"]) if os.environ.get("R1_TEACHER_DIR") else None,
        Path("/kaggle/input/output-notebook1-r1-efficientnet-b4-teacher/final_research_kd/runs/final/R1/seed_42"),
        Path("/kaggle/input/notebook1-r1-efficientnet-b4-teacher/final_research_kd/runs/final/R1"),
        Path("/kaggle/input/notebook1-r1-efficientnet-b4-teacher/runs/final/R1"),
        Path("/kaggle/input/r1-final/final_research_kd/runs/final/R1"),
        Path("/kaggle/input/r1-final/R1"),
        Path("/kaggle/input/thesis-kd-trashnet/final_research_kd/runs/final/R1"),
        Path("/kaggle/working/final_research_kd/runs/final/R1"),
        Path.cwd() / "final_research_kd" / "runs" / "final" / "R1",
    ]


def float_tag(value):
    return str(value).replace(".", "p")


cfg = Config()
if cfg.R2_VARIANT not in {"ce_baseline", "direct_kd", "twostage_kd"}:
    raise ValueError("R2_VARIANT must be 'ce_baseline', 'direct_kd', or 'twostage_kd'.")

variant_tag = cfg.R2_VARIANT.replace("_", "-")
if cfg.R2_VARIANT == "ce_baseline":
    stage_tag = f"ce{cfg.CE_EPOCHS}"
elif cfg.R2_VARIANT == "direct_kd":
    stage_tag = f"kd{cfg.KD_EPOCHS}"
else:
    stage_tag = f"ce{cfg.CE_EPOCHS}_kd{cfg.KD_EPOCHS}"
if cfg.RUN_PHASE == "pilot":
    cfg.SETUP_ID = (
        f"r2_pilot_{variant_tag}_s{cfg.SEED}_"
        f"t{float_tag(cfg.KD_TEMPERATURE)}_a{float_tag(cfg.KD_ALPHA)}_"
        f"{stage_tag}_"
        f"es-{cfg.EARLY_STOPPING_MONITOR.replace('_', '')}-p{cfg.PATIENCE}_"
        f"img{cfg.IMG_SIZE}_bs{cfg.BATCH_SIZE}"
    )
    cfg.OUTPUT_DIR = cfg.OUTPUT_ROOT / "pilots" / cfg.EXPERIMENT_ID / cfg.SETUP_ID
elif cfg.RUN_PHASE == "final":
    cfg.SETUP_ID = (
        f"r2_final_{variant_tag}_s{cfg.SEED}_"
        f"t{float_tag(cfg.KD_TEMPERATURE)}_a{float_tag(cfg.KD_ALPHA)}_"
        f"{stage_tag}_img{cfg.IMG_SIZE}_bs{cfg.BATCH_SIZE}"
    )
    cfg.OUTPUT_DIR = cfg.OUTPUT_ROOT / "final" / cfg.EXPERIMENT_ID / cfg.R2_VARIANT / f"seed_{cfg.SEED}"
else:
    raise ValueError(f"Unsupported RUN_PHASE: {cfg.RUN_PHASE}")

cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment : {cfg.EXPERIMENT_NAME}")
print(f"Run phase  : {cfg.RUN_PHASE}")
print(f"Variant    : {cfg.R2_VARIANT}")
print(f"Seed       : {cfg.SEED}")
print(f"Setup ID   : {cfg.SETUP_ID}")
print(f"CE epochs  : {cfg.CE_EPOCHS}")
print(f"KD epochs  : {cfg.KD_EPOCHS}")
print(f"T / alpha  : {cfg.KD_TEMPERATURE} / {cfg.KD_ALPHA}")
print(f"Early stop : {cfg.EARLY_STOPPING} (patience={cfg.PATIENCE})")
print(f"Eval test  : {cfg.EVALUATE_TEST}")
print(f"Output dir : {cfg.OUTPUT_DIR}")

In [ ]:
# ============================================================
# 2. Reproducibility, R0 Artifacts, and R1 Teacher
# ============================================================


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(cfg.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def has_r0_artifacts(path: Path) -> bool:
    return (
        (path / "class_mapping.json").exists()
        and (path / f"split_manifest_seed_{cfg.SEED}.csv").exists()
    )


def resolve_r0_dir(candidates):
    for candidate in candidates:
        if candidate is None:
            continue
        candidate = Path(candidate)
        if candidate.exists() and has_r0_artifacts(candidate):
            return candidate
        for nested in [candidate / "r0_data_protocol", candidate / "final_research" / "r0_data_protocol"]:
            if nested.exists() and has_r0_artifacts(nested):
                return nested
    for root in [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]:
        if root.exists():
            for path in root.rglob("r0_data_protocol"):
                if has_r0_artifacts(path):
                    return path
    raise FileNotFoundError("R0 data protocol directory not found. Set R0_DATA_PROTOCOL_DIR.")


def resolve_teacher_checkpoint(seed: int) -> Path:
    filename = f"efficientnet_b4_teacher_r1_final_seed_{seed}_best.pth"
    candidates = []
    for root in cfg.R1_TEACHER_ROOT_CANDIDATES:
        if root is None:
            continue
        root = Path(root)
        candidates.extend([
            root / f"seed_{seed}" / filename,
            root / filename,
            root / "final_research_kd" / "runs" / "final" / "R1" / f"seed_{seed}" / filename,
            root / "runs" / "final" / "R1" / f"seed_{seed}" / filename,
        ])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    for root in [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]
    raise FileNotFoundError(
        f"R1 teacher checkpoint not found for seed {seed}. "
        "Attach R1 final output or set R1_TEACHER_DIR."
    )


R0_DIR = resolve_r0_dir(cfg.R0_DIR_CANDIDATES)
CLASS_MAPPING_PATH = R0_DIR / "class_mapping.json"
MANIFEST_PATH = R0_DIR / f"split_manifest_seed_{cfg.SEED}.csv"
class_mapping = json.loads(CLASS_MAPPING_PATH.read_text(encoding="utf-8"))
manifest_df = pd.read_csv(MANIFEST_PATH)

cfg.NUM_CLASSES = int(class_mapping["num_classes"])
cfg.CLASS_NAMES = list(class_mapping["class_names"])
if set(manifest_df["seed"].unique()) != {cfg.SEED}:
    raise ValueError(f"Manifest seed mismatch. Expected only {cfg.SEED}.")

train_df = manifest_df[manifest_df["split"] == "train"].reset_index(drop=True)
val_df = manifest_df[manifest_df["split"] == "val"].reset_index(drop=True)
test_df = manifest_df[manifest_df["split"] == "test"].reset_index(drop=True)
TEACHER_CHECKPOINT_PATH = None
if cfg.R2_VARIANT in {"direct_kd", "twostage_kd"}:
    TEACHER_CHECKPOINT_PATH = resolve_teacher_checkpoint(cfg.SEED)

print(f"R0 dir: {R0_DIR}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Teacher checkpoint: {TEACHER_CHECKPOINT_PATH if TEACHER_CHECKPOINT_PATH else 'not required for CE baseline'}")
print(f"Classes: {cfg.CLASS_NAMES}")
print(f"Train/Val/Test: {len(train_df)} / {len(val_df)} / {len(test_df)}")
display(manifest_df.groupby(["split", "label"]).size().unstack(fill_value=0))

In [ ]:
# ============================================================
# 3. Dataset, Transforms, and DataLoaders
# ============================================================


class TrashNetManifestDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, dataset_dir: Path, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.dataset_dir = Path(dataset_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = Path(row["image_path"])
        if not image_path.exists():
            image_path = self.dataset_dir / row["relative_path"]
        image = np.array(Image.open(image_path).convert("RGB"))
        if self.transform is not None:
            image = self.transform(image=image)["image"]
        return image, int(row["class_id"]), row["sample_id"], row["relative_path"]


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def coarse_dropout(img_size: int):
    min_h = int(img_size * 0.05)
    max_h = int(img_size * 0.20)
    try:
        return A.CoarseDropout(
            num_holes_range=(1, 1),
            hole_height_range=(min_h, max_h),
            hole_width_range=(min_h, max_h),
            fill=0,
            p=0.5,
        )
    except TypeError:
        return A.CoarseDropout(
            max_holes=1,
            min_height=min_h,
            max_height=max_h,
            min_width=min_h,
            max_width=max_h,
            fill_value=0,
            p=0.5,
        )


train_transform = A.Compose([
    A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    coarse_dropout(cfg.IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])
eval_transform = A.Compose([
    A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])


def seed_worker(worker_id):
    worker_seed = cfg.SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


generator = torch.Generator()
generator.manual_seed(cfg.SEED)
train_loader = DataLoader(
    TrashNetManifestDataset(train_df, cfg.DATASET_DIR, train_transform),
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=generator,
)
val_loader = DataLoader(
    TrashNetManifestDataset(val_df, cfg.DATASET_DIR, eval_transform),
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    TrashNetManifestDataset(test_df, cfg.DATASET_DIR, eval_transform),
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
)
print(f"Train/Val/Test batches: {len(train_loader)} / {len(val_loader)} / {len(test_loader)}")

In [ ]:
# ============================================================
# 4. Focus-RCNet and EfficientNet-B4 Teacher
# ============================================================


class Focus(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels * 4, out_channels, kernel_size, stride=1, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(torch.cat([
            x[..., ::2, ::2],
            x[..., 1::2, ::2],
            x[..., ::2, 1::2],
            x[..., 1::2, 1::2],
        ], dim=1))


class SimAM(nn.Module):
    def __init__(self, e_lambda: float = 1e-4):
        super().__init__()
        self.e_lambda = e_lambda

    def forward(self, x):
        _, _, height, width = x.size()
        n = height * width - 1
        x_minus_mu_sq = (x - x.mean(dim=[2, 3], keepdim=True)).pow(2)
        y = x_minus_mu_sq / (
            4 * (x_minus_mu_sq.sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)
        ) + 0.5
        return x * torch.sigmoid(y)


class SandglassBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, reduction: int = 2):
        super().__init__()
        self.use_residual = stride == 1 and in_channels == out_channels
        mid_channels = max(in_channels // reduction, 1)
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, stride=stride, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(in_channels, mid_channels, 1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.Conv2d(mid_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, groups=out_channels, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        output = self.layers(x)
        return x + output if self.use_residual else output


class FocusRCNet(nn.Module):
    STAGE_CONFIG = [(48, 4, 2, 2), (96, 3, 2, 2), (192, 2, 2, 2), (384, 2, 2, 2)]

    def __init__(self, num_classes: int = 6, dropout: float = 0.2):
        super().__init__()
        self.focus = Focus(3, 24, kernel_size=1)
        stages = []
        in_channels = 24
        for out_channels, num_blocks, stride, reduction in self.STAGE_CONFIG:
            blocks = []
            for block_idx in range(num_blocks):
                block_stride = stride if block_idx == 0 else 1
                blocks.append(SandglassBlock(in_channels, out_channels, stride=block_stride, reduction=reduction))
                in_channels = out_channels
            blocks.append(SimAM())
            stages.append(nn.Sequential(*blocks))
        self.stages = nn.Sequential(*stages)
        self.conv5 = nn.Sequential(
            nn.Conv2d(in_channels, 512, kernel_size=1, bias=False),
            nn.BatchNorm2d(512),
            nn.SiLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes),
        )
        self._initialize_weights()

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, 0, 0.01)
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.focus(x)
        x = self.stages(x)
        x = self.conv5(x)
        return self.classifier(x)


def create_focus_rcnet(num_classes: int = 6):
    return FocusRCNet(num_classes=num_classes)


def create_teacher_model(pretrained: bool = False):
    return timm.create_model(cfg.TEACHER_TIMM_NAME, pretrained=pretrained, num_classes=cfg.NUM_CLASSES)


def load_teacher_model(checkpoint_path: Path):
    loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    teacher = create_teacher_model(pretrained=False)
    teacher.load_state_dict(loaded["model_state_dict"])
    teacher = teacher.to(device).eval()
    for parameter in teacher.parameters():
        parameter.requires_grad = False
    print(f"Teacher loaded: {checkpoint_path}")
    print(f"Teacher best epoch: {loaded.get('best_epoch', 'n/a')}")
    print(f"Teacher best val acc: {loaded.get('best_val_acc', 'n/a')}")
    return teacher, loaded


probe_model = create_focus_rcnet(cfg.NUM_CLASSES)
probe_params = sum(p.numel() for p in probe_model.parameters())
probe_output = probe_model(torch.randn(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE))
assert probe_params == 520_630, f"Unexpected Focus-RCNet params: {probe_params}"
assert tuple(probe_output.shape) == (1, cfg.NUM_CLASSES)
del probe_model, probe_output

teacher_model = None
teacher_checkpoint = None
if TEACHER_CHECKPOINT_PATH is not None:
    teacher_model, teacher_checkpoint = load_teacher_model(TEACHER_CHECKPOINT_PATH)
print(f"Focus-RCNet parameters: {probe_params:,}")

In [ ]:
# ============================================================
# 5. Training Helpers
# ============================================================

ce_criterion = nn.CrossEntropyLoss()


def kd_loss(student_logits, teacher_logits, labels):
    temperature = cfg.KD_TEMPERATURE
    alpha = cfg.KD_ALPHA
    student_log_soft = F.log_softmax(student_logits.float() / temperature, dim=1)
    teacher_soft = F.softmax(teacher_logits.float() / temperature, dim=1)
    loss_soft = F.kl_div(student_log_soft, teacher_soft, reduction="batchmean") * (temperature ** 2)
    loss_hard = ce_criterion(student_logits.float(), labels)
    return alpha * loss_soft + (1.0 - alpha) * loss_hard, loss_soft, loss_hard


def train_one_epoch_ce(model, loader, optimizer, scaler):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels, _, _ in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=cfg.USE_AMP):
            logits = model(images)
            loss = ce_criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        correct += logits.argmax(dim=1).eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total, None, None


def train_one_epoch_kd(student, teacher, loader, optimizer, scaler):
    student.train()
    teacher.eval()
    running_loss = 0.0
    running_soft = 0.0
    running_hard = 0.0
    correct = 0
    total = 0
    for images, labels, _, _ in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=cfg.USE_AMP):
            student_logits = student(images)
            with torch.no_grad():
                teacher_logits = teacher(images)
        with autocast(enabled=False):
            loss, loss_soft, loss_hard = kd_loss(student_logits, teacher_logits, labels)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite KD loss detected")
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        running_soft += loss_soft.item() * images.size(0)
        running_hard += loss_hard.item() * images.size(0)
        correct += student_logits.argmax(dim=1).eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total, running_soft / total, running_hard / total


@torch.no_grad()
def evaluate_loss_acc(model, loader):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels, _, _ in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast(enabled=cfg.USE_AMP):
            logits = model(images)
            loss = ce_criterion(logits, labels)
        running_loss += loss.item() * images.size(0)
        correct += logits.argmax(dim=1).eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


def monitor_improved(value, best_value):
    if cfg.EARLY_STOPPING_MONITOR == "val_loss":
        return value < best_value
    if cfg.EARLY_STOPPING_MONITOR == "val_acc":
        return value > best_value
    raise ValueError(f"Unsupported monitor: {cfg.EARLY_STOPPING_MONITOR}")


def run_stage(model, stage_name, epochs, lr, use_kd, global_epoch_start=0):
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=cfg.MOMENTUM, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler(enabled=cfg.USE_AMP)
    history = []
    best_state = None
    best_val_acc = -1.0
    best_epoch = 0
    best_monitor = float("inf") if cfg.EARLY_STOPPING_MONITOR == "val_loss" else -float("inf")
    patience_count = 0

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]
        if use_kd:
            train_loss, train_acc, loss_soft, loss_hard = train_one_epoch_kd(
                model, teacher_model, train_loader, optimizer, scaler
            )
        else:
            train_loss, train_acc, loss_soft, loss_hard = train_one_epoch_ce(
                model, train_loader, optimizer, scaler
            )
        val_loss, val_acc = evaluate_loss_acc(model, val_loader)
        scheduler.step()
        epoch_time = time.time() - epoch_start
        global_epoch = global_epoch_start + epoch

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            best_marker = " BEST"
        else:
            best_marker = ""

        monitor_value = val_loss if cfg.EARLY_STOPPING_MONITOR == "val_loss" else val_acc
        if monitor_improved(monitor_value, best_monitor):
            best_monitor = monitor_value
            patience_count = 0
        else:
            patience_count += 1

        history.append({
            "stage": stage_name,
            "epoch": epoch,
            "global_epoch": global_epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "lr": current_lr,
            "epoch_time_sec": epoch_time,
            "train_loss_soft": loss_soft,
            "train_loss_hard": loss_hard,
        })
        print(
            f"[{stage_name}] Epoch [{epoch:3d}/{epochs}] "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
            f"LR: {current_lr:.6f} | Time: {epoch_time:.1f}s{best_marker}",
            flush=True,
        )
        if cfg.EARLY_STOPPING and patience_count >= cfg.PATIENCE:
            print(f"Early stopping {stage_name} at epoch {epoch}")
            break

    return {
        "history": history,
        "best_state": best_state,
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "epochs_ran": len(history),
    }


print("Training helpers ready")

In [ ]:
# ============================================================
# 6. Run R2 Variant
# ============================================================

student_model = create_focus_rcnet(cfg.NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in student_model.parameters())
trainable_params = sum(p.numel() for p in student_model.parameters() if p.requires_grad)
print(f"Student parameters: {total_params:,}")

all_history = []
stage_summaries = []
selected_stage = None
selected_state = None
selected_best_epoch = None
selected_best_val_acc = None
total_start = time.time()

if cfg.R2_VARIANT == "direct_kd":
    kd_result = run_stage(student_model, "direct_kd", cfg.KD_EPOCHS, cfg.DIRECT_LR, use_kd=True)
    all_history.extend(kd_result["history"])
    stage_summaries.append({
        "stage": "direct_kd",
        **{k: v for k, v in kd_result.items() if k not in {"history", "best_state"}},
    })
    selected_stage = "direct_kd"
    selected_state = kd_result["best_state"]
    selected_best_epoch = kd_result["best_epoch"]
    selected_best_val_acc = kd_result["best_val_acc"]
elif cfg.R2_VARIANT == "ce_baseline":
    ce_result = run_stage(student_model, "ce_baseline", cfg.CE_EPOCHS, cfg.STAGE1_LR, use_kd=False)
    all_history.extend(ce_result["history"])
    stage_summaries.append({
        "stage": "ce_baseline",
        **{k: v for k, v in ce_result.items() if k not in {"history", "best_state"}},
    })
    selected_stage = "ce_baseline"
    selected_state = ce_result["best_state"]
    selected_best_epoch = ce_result["best_epoch"]
    selected_best_val_acc = ce_result["best_val_acc"]
elif cfg.R2_VARIANT == "twostage_kd":
    ce_result = run_stage(student_model, "ce_pretrain", cfg.CE_EPOCHS, cfg.STAGE1_LR, use_kd=False)
    all_history.extend(ce_result["history"])
    stage_summaries.append({
        "stage": "ce_pretrain",
        **{k: v for k, v in ce_result.items() if k not in {"history", "best_state"}},
    })
    if ce_result["best_state"] is None:
        raise RuntimeError("CE pretrain did not produce a checkpoint")
    student_model.load_state_dict(ce_result["best_state"])
    print(f"Loaded best CE pretrain state from epoch {ce_result['best_epoch']} before KD fine-tune")
    kd_result = run_stage(
        student_model,
        "kd_finetune",
        cfg.KD_EPOCHS,
        cfg.STAGE2_LR,
        use_kd=True,
        global_epoch_start=len(ce_result["history"]),
    )
    all_history.extend(kd_result["history"])
    stage_summaries.append({
        "stage": "kd_finetune",
        **{k: v for k, v in kd_result.items() if k not in {"history", "best_state"}},
    })
    selected_stage = "kd_finetune"
    selected_state = kd_result["best_state"]
    selected_best_epoch = kd_result["best_epoch"]
    selected_best_val_acc = kd_result["best_val_acc"]
else:
    raise ValueError(cfg.R2_VARIANT)

if selected_state is None:
    raise RuntimeError("Selected R2 state is empty")

student_model.load_state_dict(selected_state)
total_time_sec = time.time() - total_start
history_df = pd.DataFrame(all_history)
stage_summary_df = pd.DataFrame(stage_summaries)
display(stage_summary_df)
print(f"Selected stage: {selected_stage}")
print(f"Selected best epoch: {selected_best_epoch}")
print(f"Selected best val acc: {selected_best_val_acc:.6f}")
print(f"Total training time: {total_time_sec / 60:.2f} minutes")

In [ ]:
# ============================================================
# 7. Evaluation and Artifacts
# ============================================================


@torch.no_grad()
def collect_predictions(model, loader, split_name):
    model.eval()
    rows = []
    for images, labels, sample_ids, relative_paths in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast(enabled=cfg.USE_AMP):
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
        pred_ids = probs.argmax(dim=1)
        probs_np = probs.detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()
        pred_np = pred_ids.detach().cpu().numpy()
        for i in range(len(labels_np)):
            row = {
                "sample_id": sample_ids[i],
                "image_path": relative_paths[i],
                "label": cfg.CLASS_NAMES[int(labels_np[i])],
                "label_id": int(labels_np[i]),
                "prediction": cfg.CLASS_NAMES[int(pred_np[i])],
                "prediction_id": int(pred_np[i]),
                "seed": cfg.SEED,
                "model_id": cfg.EXPERIMENT_ID,
                "variant": cfg.R2_VARIANT,
                "split": split_name,
            }
            for class_idx, class_name in enumerate(cfg.CLASS_NAMES):
                row[f"prob_{class_name}"] = float(probs_np[i, class_idx])
            rows.append(row)
    return pd.DataFrame(rows)


def compute_metrics(pred_df):
    y_true = pred_df["label_id"].to_numpy()
    y_pred = pred_df["prediction_id"].to_numpy()
    prob_cols = [f"prob_{class_name}" for class_name in cfg.CLASS_NAMES]
    y_prob = pred_df[prob_cols].to_numpy()
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(cfg.NUM_CLASSES)),
        average="macro",
        zero_division=0,
    )
    try:
        auc = roc_auc_score(y_true, y_prob, labels=list(range(cfg.NUM_CLASSES)), multi_class="ovr", average="macro")
    except ValueError:
        auc = np.nan
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
        "auc_macro_ovr": auc,
    }


def plot_confusion_matrix(pred_df, path, title):
    matrix = confusion_matrix(pred_df["label_id"], pred_df["prediction_id"], labels=list(range(cfg.NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(range(cfg.NUM_CLASSES))
    ax.set_yticks(range(cfg.NUM_CLASSES))
    ax.set_xticklabels(cfg.CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(cfg.CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for row_idx in range(cfg.NUM_CLASSES):
        for col_idx in range(cfg.NUM_CLASSES):
            ax.text(col_idx, row_idx, matrix[row_idx, col_idx], ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return matrix


val_predictions = collect_predictions(student_model, val_loader, "val")
metrics_rows = [{"split": "val", **compute_metrics(val_predictions)}]
if cfg.EVALUATE_TEST and cfg.RUN_PHASE == "final":
    test_predictions = collect_predictions(student_model, test_loader, "test")
    metrics_rows.append({"split": "test", **compute_metrics(test_predictions)})
else:
    test_predictions = None
    print("Independent test evaluation skipped for pilot mode")

metrics_df = pd.DataFrame(metrics_rows)
display(metrics_df)

prefix = f"{cfg.EXPERIMENT_ID.lower()}_{cfg.R2_VARIANT}_{cfg.RUN_PHASE}_seed_{cfg.SEED}"
history_path = cfg.OUTPUT_DIR / f"training_history_{prefix}.csv"
stage_summary_path = cfg.OUTPUT_DIR / f"stage_summary_{prefix}.csv"
metrics_path = cfg.OUTPUT_DIR / f"metrics_{prefix}.csv"
val_pred_path = cfg.OUTPUT_DIR / f"predictions_{prefix}_val.csv"
checkpoint_path = cfg.OUTPUT_DIR / f"focus_rcnet_teacher_assistant_{prefix}_best.pth"
config_path = cfg.OUTPUT_DIR / f"config_{prefix}.json"
manifest_path = cfg.OUTPUT_DIR / f"artifact_manifest_{prefix}.json"
curve_path = cfg.OUTPUT_DIR / f"training_curves_{prefix}.png"

history_df.to_csv(history_path, index=False)
stage_summary_df.to_csv(stage_summary_path, index=False)
metrics_df.to_csv(metrics_path, index=False)
val_predictions.to_csv(val_pred_path, index=False)
plot_confusion_matrix(val_predictions, cfg.OUTPUT_DIR / "confusion_matrix_val.png", f"{cfg.EXPERIMENT_ID} {cfg.R2_VARIANT} validation")

artifacts = {
    "checkpoint": str(checkpoint_path),
    "config": str(config_path),
    "history": str(history_path),
    "stage_summary": str(stage_summary_path),
    "metrics": str(metrics_path),
    "val_predictions": str(val_pred_path),
    "training_curves": str(curve_path),
}
if test_predictions is not None:
    test_pred_path = cfg.OUTPUT_DIR / f"predictions_{prefix}_test.csv"
    test_predictions.to_csv(test_pred_path, index=False)
    plot_confusion_matrix(test_predictions, cfg.OUTPUT_DIR / "confusion_matrix_test.png", f"{cfg.EXPERIMENT_ID} {cfg.R2_VARIANT} test")
    artifacts["test_predictions"] = str(test_pred_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for stage_name, group in history_df.groupby("stage"):
    axes[0].plot(group["global_epoch"], group["train_loss"], label=f"{stage_name} train")
    axes[0].plot(group["global_epoch"], group["val_loss"], label=f"{stage_name} val")
    axes[1].plot(group["global_epoch"], group["train_acc"], label=f"{stage_name} train")
    axes[1].plot(group["global_epoch"], group["val_acc"], label=f"{stage_name} val")
axes[0].set_title("Loss")
axes[1].set_title("Accuracy")
for axis in axes:
    axis.set_xlabel("Global epoch")
    axis.grid(True, alpha=0.3)
    axis.legend(fontsize=8)
fig.suptitle(f"{cfg.EXPERIMENT_ID} Focus-RCNet {cfg.R2_VARIANT} - seed {cfg.SEED}")
fig.tight_layout()
fig.savefig(curve_path, dpi=160)
plt.close(fig)

config = {
    "experiment_id": cfg.EXPERIMENT_ID,
    "experiment_name": cfg.EXPERIMENT_NAME,
    "seed": cfg.SEED,
    "run_phase": cfg.RUN_PHASE,
    "r2_variant": cfg.R2_VARIANT,
    "setup_id": cfg.SETUP_ID,
    "dataset_dir": str(cfg.DATASET_DIR),
    "r0_dir": str(R0_DIR),
    "manifest_path": str(MANIFEST_PATH),
    "teacher_checkpoint_path": str(TEACHER_CHECKPOINT_PATH) if TEACHER_CHECKPOINT_PATH else None,
    "student_model_name": cfg.STUDENT_MODEL_NAME,
    "teacher_model_name": cfg.TEACHER_MODEL_NAME,
    "img_size": cfg.IMG_SIZE,
    "batch_size": cfg.BATCH_SIZE,
    "ce_epochs": cfg.CE_EPOCHS,
    "kd_epochs": cfg.KD_EPOCHS,
    "direct_lr": cfg.DIRECT_LR,
    "stage1_lr": cfg.STAGE1_LR,
    "stage2_lr": cfg.STAGE2_LR,
    "momentum": cfg.MOMENTUM,
    "weight_decay": cfg.WEIGHT_DECAY,
    "kd_temperature": cfg.KD_TEMPERATURE,
    "kd_alpha": cfg.KD_ALPHA,
    "early_stopping": cfg.EARLY_STOPPING,
    "early_stopping_monitor": cfg.EARLY_STOPPING_MONITOR,
    "patience": cfg.PATIENCE,
    "evaluate_test": cfg.EVALUATE_TEST,
    "checkpoint_metric": cfg.CHECKPOINT_METRIC,
    "selected_stage": selected_stage,
    "selected_best_epoch": selected_best_epoch,
    "selected_best_val_acc": selected_best_val_acc,
    "train_size": len(train_df),
    "val_size": len(val_df),
    "test_size": len(test_df),
    "num_classes": cfg.NUM_CLASSES,
    "class_names": cfg.CLASS_NAMES,
    "total_params": total_params,
    "trainable_params": trainable_params,
    "total_time_sec": total_time_sec,
    "stage_summaries": stage_summaries,
}

checkpoint = {
    "model_state_dict": selected_state,
    "config": config,
    "class_names": cfg.CLASS_NAMES,
    "best_epoch": selected_best_epoch,
    "best_val_acc": selected_best_val_acc,
    "selected_stage": selected_stage,
    "r2_variant": cfg.R2_VARIANT,
}
torch.save(checkpoint, checkpoint_path)
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
manifest = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment_id": cfg.EXPERIMENT_ID,
    "seed": cfg.SEED,
    "run_phase": cfg.RUN_PHASE,
    "r2_variant": cfg.R2_VARIANT,
    "setup_id": cfg.SETUP_ID,
    "output_dir": str(cfg.OUTPUT_DIR),
    "artifacts": artifacts,
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Saved checkpoint: {checkpoint_path}")
print(f"Saved config    : {config_path}")
print(f"Saved metrics   : {metrics_path}")
print(f"Saved manifest  : {manifest_path}")

In [ ]:
# ============================================================
# 8. Checkpoint Verification and Completion
# ============================================================

loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
verify_model = create_focus_rcnet(cfg.NUM_CLASSES)
verify_model.load_state_dict(loaded["model_state_dict"])
verify_model.eval()
with torch.no_grad():
    verify_logits = verify_model(torch.randn(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE))
assert tuple(verify_logits.shape) == (1, cfg.NUM_CLASSES)
print("Checkpoint verification passed")
print(f"Variant: {loaded['r2_variant']}")
print(f"Selected stage: {loaded['selected_stage']}")
print(f"Best val acc: {loaded['best_val_acc']:.6f}")
print("R2 run complete")

## Notes

Pilot R2 menunjukkan `ce_baseline` sebagai Focus-RCNet control terbaik dan
`direct_kd` T=4 alpha=0.1 sebagai kandidat KD terbaik. Final R2 dijalankan
untuk dua varian: `ce_baseline` dan `direct_kd`, masing-masing 5 seed.
Output final dipisah per varian di `runs/final/R2/<variant>/seed_<seed>/`.